# Langfuse observability for Week 2 RAG and Week 3 Agent



## 1. High-level architecture diagram


```mermaid
graph TD
    A[User query] --> B[Week 2 RAG flow]
    B --> C[Document loading]
    C --> D[Semantic chunking]
    D --> E[Embeddings]
    E --> F[Vector DB]
    F --> G[Retrieval]
    G --> H[Prompt + Bedrock LLM]
    H --> I[Answer]
    A --> J[Week 3 Agent flow]
    J --> K[Agent execution]
    K --> L[Reasoning step]
    L --> M[Tool selection]
    M --> N[DB tool]
    M --> O[Web search tool]
    N --> P[Final LLM response]
    O --> P
    P --> Q[Langfuse trace + spans]
```



## 2. What is a Trace and a Span?



- A trace is one full request or one full workflow.
- A span is one step inside that workflow.
- Example: one trace can contain spans such as document loading, chunking, retrieval, and LLM response.

What happens after running it: you will understand why we are wrapping each notebook step in a span.

## 3. — Install Langfuse

 Langfuse must be installed before we can send trace data.

What happens after running it: the package is installed in the current Python environment.

## 4.  — Set environment variables

 Langfuse needs your public key, secret key, and host so it knows where to send the data.

What happens after running it: the notebook stores the connection settings in the environment.

In [1]:
import os

os.environ['LANGFUSE_PUBLIC_KEY'] = 'pk-lf-a39cefac-6041-4ce2-a049-f2c65e357748'
os.environ['LANGFUSE_SECRET_KEY'] = 'sk-lf-1e404c55-08cc-4603-a8b5-40faf5a481fc'
os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'


## 5.  — Initialize the Langfuse client and create a trace

 this creates the connection object and the top-level trace for the notebook run.

What happens after running it: Langfuse is ready to record your steps.

In [ ]:
import langfuse
print(langfuse.__version__)

4.7.0


In [2]:
from langfuse import Langfuse
import os

langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST"),
)

with langfuse.start_as_current_observation(
    name="week2_rag_trace"
) as span:
    span.update(
        input={"notebook": "day8.ipynb"}
    )

Failed to export span batch code: 401, reason: Unauthorized


## 6. — Add Langfuse around document loading in [day8.ipynb](day8.ipynb)

this records the first major step, which is reading the PDF and collecting the text.

Insert this into the first code cell of [day8.ipynb](day8.ipynb), around the existing PDF loading logic. Keep the original PDF logic exactly the same.

What happens after running it: Langfuse records the document loading step and saves the text length as a result.

In [3]:
from pypdf import PdfReader

with langfuse.start_as_current_observation(name="document_loading") as span:
    reader = PdfReader("project_report_full.pdf")

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

    span.update(
        input={"file": "project_report_full.pdf"},
        output={"char_count": len(text)}
    )

print(text[:500])

PROJECT REPORT
Project Name: Smart City Traffic Management System
Prepared By: Michael Johnson
Role: Senior Project Manager
Organization: UrbanTech Solutions Pvt Ltd
Project ID: SCTMS-2026-001
Project Start Date: January 5, 2026
Expected Completion Date: December 20, 2026
Project Budget: $2.5 Million
Location: Hyderabad, India
Project Objective
The Smart City Traffic Management System aims to reduce traffic congestion and improve road
safety through the use of artificial intelligence, real-time 


## 7. — Add Langfuse around semantic chunking in [day8.ipynb](day8.ipynb)

semantic chunking is a separate stage, so it should be its own span.

Insert this into the chunking code cell of [day8.ipynb](day8.ipynb). The chunking logic stays the same.

What happens after running it: Langfuse records how many chunks were created.

In [5]:
import spacy
from sentence_transformers import SentenceTransformer
import chromadb

nlp = spacy.load("en_core_web_sm")

def semantic_chunking(text, max_len=256):
    doc = nlp(text)
    chunks = []
    current_chunk = ""

    for sent in doc.sents:
        if len(current_chunk) + len(sent.text) <= max_len:
            current_chunk += " " + sent.text
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sent.text

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

with langfuse.start_as_current_observation(name="semantic_chunking") as span:
    chunks = semantic_chunking(text, max_len=256)

    span.update(
        output={
            "chunk_count": len(chunks),
            "preview": chunks[:3]
        }
    )

print("Number of semantic chunks:", len(chunks))

Number of semantic chunks: 13


Failed to export span batch code: 401, reason: Unauthorized


## 8. — Add Langfuse around embedding generation and vector DB insertion in [day8.ipynb](day8.ipynb)

 embeddings and database insertion are important steps, and they should be visible separately.

Insert this into the cell that creates the embedding model and writes chunks into Chroma.

What happens after running it: Langfuse records the embedding step and the number of chunks added to the vector store.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.get_or_create_collection("project_report")

with langfuse.start_as_current_observation(name="embedding_and_vector_db_insert") as span:

    for i, chunk in enumerate(chunks):
        embedding = model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[f"chunk_{i}"],
            metadatas=[{"section": "semantic"}]
        )

    span.update(
        output={
            "chunks_added": len(chunks)
        }
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Failed to export span batch code: 401, reason: Unauthorized


## 9. — Add Langfuse around retrieval in [day8.ipynb](day8.ipynb)

 retrieval is the moment where the vector database is searched using the user query.

Insert this into the retrieval cell of [day8.ipynb](day8.ipynb).

What happens after running it: Langfuse records the retrieved documents from the vector DB.

## 10.  — Add Langfuse around prompt creation and LLM response generation in [day8.ipynb](day8.ipynb)

this records the prompt that is sent to the model and the response that comes back.

Insert this into the prompt and Bedrock invocation cell of [day8.ipynb](day8.ipynb).

What happens after running it: Langfuse shows the input prompt, the model response, and any token usage if available.

In [ ]:
import json
import os
import boto3
from dotenv import load_dotenv

load_dotenv("myenv.env")

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

# Set the query and retrieve relevant documents from the vector store.
if "query" not in globals():
    query = "What are the traffic management strategies in 2024?"

if "model" not in globals():
    from sentence_transformers import SentenceTransformer
    import chromadb

    model = SentenceTransformer("all-MiniLM-L6-v2")
    client = chromadb.Client()
    collection = client.get_or_create_collection("project_report")

if "collection" not in globals():
    import chromadb

    client = chromadb.Client()
    collection = client.get_or_create_collection("project_report")

try:
    query_embedding = model.encode([query])[0]
    results_internal = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
    )
    print("Internal results:", results_internal["documents"])
except Exception as exc:
    results_internal = {
        "documents": [[f"Retrieval failed: {exc}"]]
    }
    print("Internal results:", results_internal["documents"])

# Initialize Bedrock runtime for the LLM call.
if "bedrock_runtime" not in globals():
    try:
        bedrock_runtime = boto3.client(
            service_name="bedrock-runtime",
            region_name=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        )
    except Exception as exc:
        bedrock_runtime = None
        bedrock_error = str(exc)

if "MODEL_ID" not in globals():
    MODEL_ID = "amazon.nova-micro-v1:0"

if bedrock_runtime is None:
    answer = (
        "Bedrock runtime could not be initialized. "
        "Check your AWS credentials in myenv.env or run the day8 setup cells first."
    )
else:
    prompt = f"""
Context:
{chr(10).join(results_internal['documents'][0])}

Question:
{query}

Answer based on the context. If partially available,
summarize whatever relevant information you can find.
"""

    body = {
        "messages": [
            {
                "role": "user",
                "content": [{"text": prompt}]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 512,
            "temperature": 0.2
        }
    }

    if "langfuse" in globals():
        with langfuse.start_as_current_observation(name="prompt_creation") as prompt_span:
            prompt_span.update(
                input={"query": query},
                output={"prompt_preview": prompt[:400]}
            )

        with langfuse.start_as_current_observation(name="llm_response_generation") as llm_span:
            response = bedrock_runtime.invoke_model(
                modelId=MODEL_ID,
                body=json.dumps(body),
                contentType="application/json",
                accept="application/json",
            )

            response_data = json.loads(response["body"].read())
            answer = response_data["output"]["message"]["content"][0]["text"]

            llm_span.update(
                input={"prompt": prompt},
                output={"answer_preview": answer[:400]}
            )
    else:
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps(body),
            contentType="application/json",
            accept="application/json",
        )

        response_data = json.loads(response["body"].read())
        answer = response_data["output"]["message"]["content"][0]["text"]

print(answer)

Internal results: [['Conclusion\nThe Smart City Traffic Management System is progressing according to schedule and remains\nwithin the approved budget. The project team expects successful completion by December 2026.', 'PROJECT REPORT\nProject Name: Smart City Traffic Management System\nPrepared By: Michael Johnson\nRole: Senior Project Manager\nOrganization: UrbanTech Solutions Pvt Ltd\nProject ID: SCTMS-2026-001\nProject Start Date: January 5, 2026\nExpected Completion Date: December 20, 2026\nProject Budget: $2.5 Million\nLocation:', 'Hyderabad, India\nProject Objective\nThe Smart City Traffic Management System aims to reduce traffic congestion and improve road\nsafety through the use of artificial intelligence, real-time traffic monitoring, and predictive analytics.']]
Based on the provided context, specific details about traffic management strategies in 2024 are not explicitly mentioned. The context focuses on the current project, the Smart City Traffic Management System, which is

Failed to export span batch code: 401, reason: Unauthorized


## 11.  Add Langfuse around agent execution in [day12.ipynb](day12.ipynb)
 the agent flow is a full workflow, so it should have a parent span around the whole execution.

Insert this into the Agent function in [day12.ipynb](day12.ipynb). The original agent logic stays the same.

What happens after running it: Langfuse records the full agent run and the answer it produced.

In [ ]:
def Agent(question):
    with langfuse.start_as_current_observation(name="agent_execution") as agent_span:

        agent_span.update(
            input={"question": question}
        )

        response = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": question
                    }
                ]
            }
        )

        answer = response["messages"][-1].content

        agent_span.update(
            output={
                "answer_preview": answer[:400]
            }
        )

        return answer

Failed to export span batch code: 401, reason: Unauthorized


## 12. Add Langfuse around reasoning and tool selection in [day12.ipynb](day12.ipynb)
 the agent decides what to do next, so this is a very useful place to observe.

Insert this into the main agent wrapper part of [day12.ipynb](day12.ipynb) if you want more detail about the reasoning stage.

What happens after running it: Langfuse records the reasoning and tool selection stage as a child span.

In [ ]:
with langfuse.start_as_current_observation(name="reasoning_step") as reasoning_span:

    reasoning_span.update(
        input={
            "question": "who manages employee 101 and work as manager?"
        }
    )

    reasoning_span.update(
        output={
            "decision": "Agent will decide whether to use DB tool or web search"
        }
    )

Failed to export span batch code: 401, reason: Unauthorized


In [12]:
# Flush all pending observations to Langfuse Cloud
langfuse.flush()

print("All Langfuse observations sent successfully!")

All Langfuse observations sent successfully!


## 13. Add Langfuse around the database tool in [day12.ipynb](day12.ipynb)

the database tool is a separate action, so it should have its own clear span.

Insert this into the db_query_tool function in [day12.ipynb](day12.ipynb).

What happens after running it: Langfuse records the database lookup and its result.

In [13]:
from langchain_core.tools import tool

@tool
def db_query_tool(employee_id: int) -> str:
    """Retrieve employee information from the database."""

    with langfuse.start_as_current_observation(name="database_tool") as db_span:
        db_span.update(input={"employee_id": employee_id})

        try:
            result = db_query_retry(employee_id)

            db_span.update(output={"result": result})
            return result

        except Exception as e:
            db_span.update(
                output={"error": str(e)}
            )
            raise

## 14.  Add Langfuse around the web search tool in [day12.ipynb](day12.ipynb)

 the web search tool is another independent action, and it is useful to see separately.

Insert this into the web_search_tool function in [day12.ipynb](day12.ipynb).

What happens after running it: Langfuse records the search query and the returned search result.

In [14]:
from langchain_core.tools import tool

@tool
def web_search_tool(query: str) -> str:
    """Search the internet for information."""

    with langfuse.start_as_current_observation(name="web_search_tool") as web_span:

        web_span.update(input={"query": query})

        try:
            result = web_search_retry(query)

            web_span.update(output={"result": result})

            return result

        except Exception as e:
            web_span.update(output={"error": str(e)})
            raise

## 15. Add Langfuse around the final LLM response and completion in [day12.ipynb](day12.ipynb)

 this captures the final answer and closes the observation nicely.

Insert this into the final output part of the agent flow, just before the final print statement.

What happens after running it: Langfuse records the final answer and the entire agent workflow completes.

In [16]:
with langfuse.start_as_current_observation(name="final_llm_response") as final_span:
    final_span.update(
        output={"final_answer": answer}
    )

print(answer)

# Send all observations to Langfuse
langfuse.flush()

print("All Langfuse observations sent successfully!")

Based on the provided context, specific details about the traffic management strategies in 2024 are not explicitly mentioned. The context focuses on the current project, the Smart City Traffic Management System, which is set to be completed by December 2026. The project aims to reduce traffic congestion and improve road safety using artificial intelligence, real-time traffic monitoring, and predictive analytics.

However, we can infer that the strategies planned for the Smart City Traffic Management System likely include advanced technologies such as AI for predictive analytics, real-time monitoring systems to manage traffic flow, and potentially other innovative methods to optimize traffic management. These strategies are intended to be implemented as part of the project's completion in December 2026.

For detailed information on traffic management strategies in 2024, additional context or documentation from earlier phases or other projects would be required.


Failed to export span batch code: 401, reason: Unauthorized


All Langfuse observations sent successfully!
